# Chapter 6 — Storage and Volumes

This chapter solves one of the biggest real-world problems in Kubernetes:

> What happens to data when pods die?

Until now:

* we focused on running applications,
* scaling,
* networking,
* self-healing.

But real applications also need:

* databases,
* uploaded files,
* logs,
* user data,
* persistent storage.

This chapter explains how Kubernetes handles data safely.

---

# City Analogy Expansion

| Kubernetes                    | City Analogy                       |
| ----------------------------- | ---------------------------------- |
| Pod                           | Apartment building                 |
| Container filesystem          | Temporary furniture inside flat    |
| Volume                        | Shared storage room                |
| Persistent Volume (PV)        | Permanent warehouse                |
| Persistent Volume Claim (PVC) | Warehouse rental request           |
| Storage Class                 | Warehouse type/category            |
| Stateful Application          | Business needing permanent records |

---

# The Big Problem

Pods are temporary.

Remember:

* Kubernetes can destroy pods anytime,
* move them,
* recreate them.

What happens if database data is stored INSIDE pod?

It disappears.

---

# Example Disaster

Suppose:

* MySQL database runs inside pod,
* pod crashes,
* Kubernetes creates new pod.

If data was inside pod filesystem:

```text id="7n42a2"
ALL DATA LOST
```

Very dangerous.

---

# City Analogy

Imagine:

* all business records stored inside temporary apartment buildings.

If building collapses:

* records vanish.

Cities instead use:

> permanent warehouses and archives.

That’s Kubernetes persistent storage.

---

# Part 1 — Container Filesystem Is Temporary

Every container has:

* writable filesystem.

But:

> it is ephemeral.

Meaning:

* tied to container lifecycle.

When container dies:

* filesystem disappears.

---

# Example

Inside container:

```text id="x01ht8"
save file → /data/orders.txt
```

Container crashes.

New container starts.

File gone.

---

# City Analogy

Furniture inside rented flat disappears when flat destroyed.

Not permanent storage.

---

# Important Kubernetes Philosophy

Containers should ideally be:

> stateless.

Meaning:

* no critical permanent data stored internally.

Why?
Because Kubernetes treats pods as replaceable.

---

# But Real Apps Need Storage

Examples:

* databases,
* uploaded images,
* machine learning models,
* logs,
* backups.

So Kubernetes introduces:

> Volumes.

---

# Part 2 — What Is a Volume?

A Volume is:

> storage mounted into a pod.

Volume lifetime can be:

* temporary,
* or persistent.

---

# City Analogy

Volume = shared storage room connected to apartment building.

Residents can:

* store files,
* access shared resources.

---

# Important Understanding

Volume exists separately from container lifecycle.

Containers may restart.
Volume can survive.

---

# Visual Mental Model

```text id="3ot2u9"
Pod
 ├── Container A
 ├── Container B
 └── Shared Volume
```

All containers inside pod can access volume.

---

# Example Use Case

Suppose:

* app container generates logs,
* logging sidecar collects logs.

Both access same volume.

---

# Part 3 — EmptyDir Volume

Simplest volume type.

Created when pod starts.
Deleted when pod dies.

---

# Use Cases

Good for:

* cache,
* temporary files,
* inter-container sharing.

Bad for:

* databases,
* permanent storage.

---

# City Analogy

Temporary storage room rented only while building exists.

Building demolished?
Storage disappears too.

---

# Part 4 — Persistent Storage Problem

Suppose pod moves to another node.

Storage must:

* survive pod recreation,
* survive node failures,
* remain accessible.

This is MUCH harder in distributed systems.

---

# Kubernetes Solution

Kubernetes separates:

* compute,
  FROM
* storage.

Very important architecture decision.

---

# Part 5 — Persistent Volume (PV)

Persistent Volume is:

> Actual storage resource in cluster.

Could be:

* cloud disk,
* SSD,
* NFS,
* storage appliance.

---

# City Analogy

PV = permanent warehouse in city.

Independent of apartment buildings.

Buildings may change.
Warehouse remains.

---

# Important Understanding

PV exists independently from pods.

Pods use PVs.

Pods do NOT own PVs.

---

# Visual Mental Model

```text id="mylc3h"
Pod
   ↓ uses
Persistent Volume
```

---

# Part 6 — Persistent Volume Claim (PVC)

PVC is:

> A request for storage.

Application says:

```text id="r5e2i9"
I need:
- 100GB
- fast SSD
- read/write access
```

Kubernetes finds matching PV.

---

# City Analogy

Business submits warehouse rental request:

* required size,
* storage type,
* access needs.

City allocates matching warehouse.

---

# Important Separation

---

## PV

Actual storage resource.

---

## PVC

Request for storage.

---

# Why This Separation Is Powerful

Applications don’t need to know:

* actual disk location,
* cloud provider details,
* storage implementation.

Kubernetes abstracts it.

---

# Part 7 — Storage Class

StorageClass defines:

> HOW storage should be created.

Examples:

* SSD,
* HDD,
* high-performance storage,
* cloud disks.

---

# City Analogy

Warehouse categories:

* cold storage,
* premium secure warehouse,
* cheap storage,
* high-speed logistics center.

---

# Example

Database may require:

```text id="d8o4rm"
fast SSD storage
```

Backup system may use:

```text id="hq4l0w"
cheap slow storage
```

---

# Part 8 — Dynamic Provisioning

Very important modern Kubernetes feature.

Earlier:

* admins manually created storage.

Now:

* PVC automatically triggers storage creation.

---

# Example Flow

Application requests:

```text id="cdpcl2"
500GB SSD
```

StorageClass automatically:

* creates cloud disk,
* attaches volume,
* binds PVC.

---

# City Analogy

Business requests warehouse.

City automatically constructs one if unavailable.

---

# This Is Huge for Cloud Computing

Cloud platforms like:

* Amazon Web Services
* Google Cloud
* Microsoft Azure

automatically provision storage through Kubernetes.

---

# Part 9 — Stateful vs Stateless Applications

This is EXTREMELY important.

---

# Stateless Applications

Examples:

* frontend apps,
* APIs,
* workers.

If pod dies:

* no important data lost.

Easy to scale.

---

# Stateful Applications

Examples:

* databases,
* Kafka,
* Elasticsearch.

They maintain important persistent data.

Storage becomes critical.

---

# City Analogy

---

## Stateless Business

Food delivery office.
Can move buildings easily.

---

## Stateful Business

Bank archive office.
Must preserve records permanently.

---

# Why Stateful Systems Are Hard

Distributed storage introduces:

* consistency problems,
* replication complexity,
* failover issues,
* performance challenges.

This is one reason Kubernetes is considered advanced.

---

# Part 10 — Access Modes

Volumes support different access patterns.

---

# Common Modes

---

## ReadWriteOnce (RWO)

One node can mount volume.

Common for databases.

---

## ReadOnlyMany (ROX)

Multiple nodes read-only.

---

## ReadWriteMany (RWX)

Multiple nodes can read/write.

Harder to implement.

---

# City Analogy

Warehouse access permissions:

* single business,
* multiple viewers,
* shared editing access.

---

# Part 11 — StatefulSets (Introduction)

Deployments work great for stateless apps.

But databases often need:

* stable identity,
* ordered startup,
* persistent storage mapping.

Kubernetes provides:

> StatefulSets.

We’ll deeply cover later.

---

# Example

Database replicas may require:

```text id="n9v4t1"
db-0
db-1
db-2
```

Stable identities matter.

---

# Why Storage Is Hard in Kubernetes

Kubernetes was originally optimized for:

* stateless container workloads.

Persistent distributed storage is inherently difficult because:

* pods move,
* nodes fail,
* networks partition.

Storage systems are much harder than compute orchestration.

---

# Real Production Example

Suppose:

* Spotify runs recommendation databases.

Even if:

* pod crashes,
* node fails,
* cluster scales,

user playlists must survive.

Persistent Volumes enable this.

---

# Internal Storage Flow

---

# Step 1 — Application Creates PVC

```text id="krp84u"
Need 1TB SSD
```

---

# Step 2 — Kubernetes Finds StorageClass

Determines storage type.

---

# Step 3 — PV Provisioned

Cloud disk or storage created.

---

# Step 4 — Pod Mounts Volume

Container accesses storage.

---

# Visual Flow

```text id="n3wzmh"
Pod
  ↓ mounts
PVC
  ↓ binds
PV
  ↓ backed by
Actual Storage
```

---

# Important Beginner Mistakes

---

## Mistake 1

Storing database data inside container filesystem.

Dangerous.

---

## Mistake 2

Thinking pods are permanent storage units.

Pods are temporary compute units.

---

## Mistake 3

Confusing PVC and PV.

Remember:

* PVC = request
* PV = actual storage

---

# Kubernetes Core Philosophy Again

Kubernetes separates:

* compute,
* networking,
* storage.

Each managed independently.

This modularity creates flexibility and scalability.

---

# Final Mental Model

---

## Pod

Temporary execution building.

---

## Volume

Attached storage room.

---

## Persistent Volume

Permanent warehouse.

---

## Persistent Volume Claim

Warehouse rental request.

---

## StorageClass

Warehouse category/type.

---

# Full Storage Hierarchy

```text id="1i0n4j"
Pod
  ↓
Volume
  ↓
PVC
  ↓
PV
  ↓
Physical/Cloud Storage
```

---

# Most Important Insights From Chapter 6

---

## 1. Pods are temporary

Never trust pod-local storage for critical data.

---

## 2. Volumes separate storage from containers

Storage survives container restarts.

---

## 3. PV and PVC abstract storage management

Applications request storage declaratively.

---

## 4. Stateful systems are harder

Persistent distributed data is complex.

---

## 5. Kubernetes separates compute and storage

This enables portability and scalability.

---

# One-Line Summary

> Kubernetes Persistent Volumes provide durable storage independent of temporary pods, enabling stateful applications to survive crashes, scaling, and node failures.

---

